# FraudTransformer — full-data training (Kaggle T4)

Trains the Rhea FinGraph FraudTransformer (GPT-style causal transformer with
temporal-interval embeddings) on the FULL 14.63M-row train split, evaluates
on the locked 4.88M test split, and writes a model + `model_config.json`
back to the repo so `make promote-fraud-transformer` can complete locally.

The local CPU smoke (897 sequences, val AUC 0.5532) only sanity-checks the
architecture. THIS notebook is the source of the honest full-data number.

**Dataset**: IBM fraud transactions (already split chronologically into
`ibm_full/train|validation|test.parquet`). Fraud rate ~0.12%.

**Anti-fragility built in**: strictly chronological per-entity sequences,
focal loss (alpha=0.45, gamma=2.0), dropout 0.25, layer-norm, AdamW
weight-decay 0.01, early stopping on the fraud-dense val band, and the
locked test split shares the SAME boundary as every other model so the
FraudTransformer ROC is directly comparable.

In [ ]:
# ---- 0. Mount the repo so the package is importable ----
import os, sys
REPO = "/kaggle/input/rhea-fingraph"  # adjust: upload the repo (or its src) as a Kaggle dataset
sys.path.insert(0, os.path.join(REPO, "src"))
os.chdir(REPO)
print("cwd", os.getcwd())

In [ ]:
# ---- 1. Verify data + runtime ----
from pathlib import Path
import torch, polars as pl
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
base = Path("data/processed/ibm_full")
for p in ("train.parquet", "validation.parquet", "test.parquet"):
    df = pl.scan_parquet(base/p).select(pl.len(), pl.col("is_fraud").sum().alias("frauds"))
    print(p, df.collect().row(0))

In [ ]:
# ---- 2. Full-data training on the T4 ----
# The local CLI already implements everything; run it uncapped (--limit None)
# so it computes the locked test ROC.
!OMP_NUM_THREADS=8 python -m fingraph_sentinel.train_fraud_transformer \
    --device cuda \
    --epochs 12 \
    --batch-size 512 \
    --max-len 64 \
    --lr 3e-4 \
    --out artifacts/models/fraud-transformer-full

# NOTE: if the IBM train.parquet is too big to frame in RAM on the T4 (14.6M
# rows), first run `python -m fingraph_sentinel.dataset ... --write-splits` on
# a CPU machine to emit the splits, then stream them here. The trainer frames
# per-customer sequences which are memory-light vs raw rows.

In [ ]:
# ---- 3. Inspect the locked result ----
import json
cfg = json.load(open("artifacts/models/fraud-transformer-full/model_config.json"))
print("val", cfg["metrics_validation"])
print("test", cfg["metrics_test_locked"])
print(json.dumps(cfg['history'], indent=1))

In [ ]:
# ---- 4. (Optional) Fuse with the velocity/v3 ensemble ----
# Export per-row FraudTransformer probabilities to glob together with the
# existing XGBoost / velocity / GNN scores into `ensemble_fusion.py`.
print("See docs/FUSION_KAGGLE_RUNBOOK.md for the ensemble glue.")